In [ ]:
# Reading the CSV data
transactions_df = spark.read.format("csv").option("header","True").option("inferSchema","True").load("/Volumes/workspace/default/financial_raw/transactions_sample_500k.csv")
users_df = spark.read.format("csv").option("header","True").option("inferSchema","True").load("/Volumes/workspace/default/financial_raw/users_raw.csv")
cards_df = spark.read.format("csv").option("header","True").option("inferSchema","True").load("/Volumes/workspace/default/financial_raw/cards_raw.csv")

transactions_df.show(5)
display(users_df.limit(5))
display(cards_df.limit(5))

In [ ]:
# Checking the schema
transactions_df.printSchema()
users_df.printSchema()
cards_df.printSchema()

In [ ]:
# Counting the rows in each DataFrame
display(transactions_df.count())
display(users_df.count())
display(cards_df.count())

In [ ]:
# Data Quality validation for each Dataframe
from pyspark.sql.functions import *
transactions_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in transactions_df.columns
]).show()
users_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in users_df.columns]).show()
cards_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in cards_df.columns]).show()

In [ ]:
# Checking for Duplicates
print("total transaction rows :", transactions_df.count())
print("distinct transaction rows :", transactions_df.distinct().count())

print("total user rows :" , users_df.count())
print("distinct user rows :", users_df.distinct().count())

print("total card rows :",cards_df.count())
print("distinct card rows :", cards_df.distinct().count())

In [ ]:
# Check the number of columns
print("Transaction columns:", len(transactions_df.columns))
print("User columns:", len(users_df.columns))
print("Card columns:", len(cards_df.columns))

# Display column names
print(transactions_df.columns)
print(users_df.columns)
print(cards_df.columns)

In [ ]:
# Writing into Parquet Files
transactions_df.write.mode("overwrite").parquet("/Volumes/workspace/default/financial_raw/parquet/transactions")
users_df.write.mode("overwrite").parquet("/Volumes/workspace/default/financial_raw/parquet/users")
cards_df.write.mode("overwrite").parquet("/Volumes/workspace/default/financial_raw/parquet/cards")
print("Parquet files written sucessfully")

In [ ]:
display(dbutils.fs.ls("/Volumes/workspace/default/financial_raw/parquet/transactions"))
display(dbutils.fs.ls("/Volumes/workspace/default/financial_raw/parquet/users"))
display(dbutils.fs.ls("/Volumes/workspace/default/financial_raw/parquet/cards"))

In [ ]:
transactions_parquet_df = spark.read.parquet("/Volumes/workspace/default/financial_raw/parquet/transactions")
transactions_parquet_df.show(3)
transactions_parquet_df.printSchema()

In [ ]:
import re

def clean_columns(df):
    for c in df.columns:
        new_col = re.sub(r'[^a-zA-Z0-9]', '_', c).lower()
        new_col = re.sub(r'_+', '_', new_col).strip('_')
        df = df.withColumnRenamed(c, new_col)
    return df

transactions_bronze_df = clean_columns(transactions_parquet_df)
users_bronze_df = clean_columns(users_df)
cards_bronze_df = clean_columns(cards_df)

transactions_bronze_df.printSchema()

In [ ]:
# Creating bronze transactions  Delta Table
transactions_bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_transactions")
users_bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_users")
cards_bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_cards")

In [ ]:
spark.table("workspace.default.bronze_transactions").show(3)

In [ ]:
spark.sql(" SHOW TABLES IN workspace.default").show()